# EB-JEPA Planning Eval

Loads the prescribed and free checkpoints from Drive, runs the planning eval (MPPI, 20 episodes), and saves the success rate.


In [ ]:
# CELL 1: Setup
!pip install -q einops fire omegaconf ruamel.yaml pymunk imageio seaborn

import os, torch
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
print(f'Conditions: {os.listdir(DRIVE_BASE)}')

In [ ]:
# CELL 2: Extract eb_jepa
import zipfile, os
EB_DIR = '/content/eb_jepa'
if os.path.exists(os.path.join(EB_DIR, 'eb_jepa', 'jepa.py')):
    print('Already extracted')
else:
    from google.colab import files
    print('Upload eb_jepa_colab.zip:')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    with zipfile.ZipFile(fname, 'r') as zf:
        zf.extractall(EB_DIR)
    print('Extracted')
os.chdir(EB_DIR)
!pip install -e . -q
from eb_jepa.jepa import JEPA
print('OK')

In [ ]:
# CELL 3: Write planning_eval.py
import os
script_dir = '/content/eb_jepa/experiments/prescribed_axes'
os.makedirs(script_dir, exist_ok=True)
script_path = os.path.join(script_dir, 'planning_eval.py')

SCRIPT = '"""\nPlanning eval for EB-JEPA v3 — runs on Colab GPU.\n\nLoads checkpoints from Drive, runs planning eval (success rate),\nsaves results back to Drive.\n\nHandles both free (ImpalaEncoder) and prescribed (PrescribedEncoder) models.\nFor prescribed: patches GCAgent to pass locations from env.info to encoder.\n"""\n\nimport copy\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport yaml\nfrom omegaconf import OmegaConf\nfrom tqdm import tqdm\n\n# EB-JEPA imports\nfrom eb_jepa.architectures import (\n    ImpalaEncoder, InverseDynamicsModel, Projector, RNNPredictor,\n)\nfrom eb_jepa.datasets.utils import init_data\nfrom eb_jepa.jepa import JEPA, JEPAProbe\nfrom eb_jepa.logging import get_logger\nfrom eb_jepa.losses import SquareLossSeq, VC_IDM_Sim_Regularizer\nfrom eb_jepa.schedulers import CosineWithWarmup\nfrom eb_jepa.state_decoder import MLPXYHead\nfrom eb_jepa.training_utils import load_config, setup_device, setup_seed\nfrom eb_jepa.planning import GCAgent, main_eval\n\nlogger = get_logger(__name__)\n\nDEVICE = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n\n\n# ================================================================\n# Encoders (same as training script)\n# ================================================================\n\nclass PrescribedEncoder(nn.Module):\n    def __init__(self, prescribed_dim=2, output_dim=512, hidden_dim=256, final_ln=True):\n        super().__init__()\n        self.mlp_output_dim = output_dim\n        self.prescribed_dim = prescribed_dim\n        self.projection = nn.Sequential(\n            nn.Linear(prescribed_dim, hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, hidden_dim),\n            nn.ReLU(),\n            nn.Linear(hidden_dim, output_dim),\n        )\n        self.final_ln = nn.LayerNorm(output_dim) if final_ln else nn.Identity()\n\n    def forward(self, observations, locations=None):\n        if locations is None:\n            raise ValueError("PrescribedEncoder requires locations")\n        loc = locations.permute(0, 2, 1)\n        B, T, _ = loc.shape\n        features = self.projection(loc.reshape(B * T, -1))\n        features = self.final_ln(features)\n        features = features.reshape(B, T, -1)\n        return features.transpose(1, 2).unsqueeze(-1).unsqueeze(-1)\n\n\nclass HybridEncoder(nn.Module):\n    def __init__(self, pixel_encoder, prescribed_dim=2, prescribed_output_dim=128, final_ln=True):\n        super().__init__()\n        self.pixel_encoder = pixel_encoder\n        self.prescribed_dim = prescribed_dim\n        total_dim = pixel_encoder.mlp_output_dim\n        free_dim = total_dim - prescribed_output_dim\n        self.prescribed_output_dim = prescribed_output_dim\n        self.free_dim = free_dim\n        self.mlp_output_dim = total_dim\n        self.prescribed_projection = nn.Sequential(\n            nn.Linear(prescribed_dim, prescribed_output_dim),\n            nn.ReLU(),\n            nn.Linear(prescribed_output_dim, prescribed_output_dim),\n        )\n        self.pixel_reduction = nn.Linear(pixel_encoder.mlp_output_dim, free_dim)\n        self.final_ln = nn.LayerNorm(total_dim) if final_ln else nn.Identity()\n\n    def forward(self, observations, locations=None):\n        if locations is None:\n            raise ValueError("HybridEncoder requires locations")\n        pixel_features = self.pixel_encoder(observations)\n        B, D, T, _, _ = pixel_features.shape\n        pixel_feat = pixel_features.squeeze(-1).squeeze(-1).transpose(1, 2)\n        pixel_feat = self.pixel_reduction(pixel_feat)\n        loc = locations.permute(0, 2, 1)\n        loc_flat = loc.reshape(B * T, -1)\n        prescribed_feat = self.prescribed_projection(loc_flat).reshape(B, T, -1)\n        combined = torch.cat([prescribed_feat, pixel_feat], dim=-1)\n        combined = self.final_ln(combined)\n        return combined.transpose(1, 2).unsqueeze(-1).unsqueeze(-1)\n\n\nclass PrescribedJEPA(JEPA):\n    def __init__(self, encoder, aencoder, predictor, regularizer, predcost):\n        super().__init__(encoder, aencoder, predictor, regularizer, predcost)\n        self._current_locations = None\n\n    def set_locations_for_planning(self, loc):\n        self._current_locations = loc\n\n    def clear_planning_locations(self):\n        self._current_locations = None\n\n    @torch.no_grad()\n    def encode(self, observations):\n        if self._current_locations is not None and hasattr(self.encoder, \'prescribed_dim\'):\n            loc = self._current_locations\n            # Expand locations to match observation batch size\n            B = observations.shape[0]\n            if loc.shape[0] == 1 and B > 1:\n                loc = loc.expand(B, -1, -1)\n            return self.encoder(observations, locations=loc)\n        if hasattr(self.encoder, \'prescribed_dim\'):\n            raise ValueError("PrescribedEncoder.encode() needs locations.")\n        return self.encoder(observations)\n\n    def unroll(self, observations, actions, nsteps=1, unroll_mode="parallel",\n               ctxt_window_time=1, compute_loss=True, return_all_steps=False,\n               locations=None):\n        if locations is not None and hasattr(self.encoder, \'prescribed_dim\'):\n            B = observations.shape[0]\n            if locations.shape[0] == 1 and B > 1:\n                locations = locations.expand(B, -1, -1)\n            state = self.encoder(observations, locations=locations)\n        elif self._current_locations is not None and hasattr(self.encoder, \'prescribed_dim\'):\n            loc = self._current_locations\n            B = observations.shape[0]\n            if loc.shape[0] == 1 and B > 1:\n                loc = loc.expand(B, -1, -1)\n            state = self.encoder(observations, locations=loc)\n        else:\n            state = self.encoder(observations)\n\n        context_length = getattr(self.predictor, "context_length", 0)\n\n        if compute_loss:\n            rloss, rloss_unweight, rloss_dict = self.regularizer(state, actions)\n            ploss = 0.0\n        else:\n            rloss = rloss_unweight = rloss_dict = ploss = None\n\n        actions_encoded = self.action_encoder(actions) if actions is not None else None\n        all_steps = [] if return_all_steps else None\n\n        if unroll_mode == "parallel":\n            predicted_states = state\n            for _ in range(nsteps):\n                predicted_states = self.predictor(predicted_states, actions_encoded)[:, :, :-1]\n                if return_all_steps:\n                    all_steps.append(predicted_states)\n                predicted_states = torch.cat(\n                    (state[:, :, :context_length], predicted_states), dim=2\n                )\n                if compute_loss:\n                    ploss += self.predcost(state, predicted_states) / nsteps\n\n        elif unroll_mode == "autoregressive":\n            if actions is not None and nsteps > actions.size(2):\n                raise ValueError(f"nsteps ({nsteps}) > actions ({actions.size(2)})")\n            effective_ctxt_window = 1 if self.single_unroll else ctxt_window_time\n            predicted_states = state[:, :, :effective_ctxt_window]\n            for i in range(nsteps):\n                context_states = predicted_states[:, :, -effective_ctxt_window:]\n                if actions_encoded is not None:\n                    context_actions = actions_encoded[\n                        :, :, max(0, i + 1 - effective_ctxt_window): i + 1\n                    ]\n                else:\n                    context_actions = None\n                pred_step = self.predictor(context_states, context_actions)[:, :, -1:]\n                predicted_states = torch.cat([predicted_states, pred_step], dim=2)\n                if return_all_steps:\n                    all_steps.append(predicted_states.clone())\n                if compute_loss:\n                    ploss += torch.nn.functional.mse_loss(\n                        pred_step, state[:, :, i + 1: i + 2]\n                    ) / nsteps\n        else:\n            raise ValueError(f"Unknown unroll_mode: {unroll_mode}")\n\n        if compute_loss:\n            losses = (ploss + rloss, rloss, rloss_unweight, rloss_dict, ploss)\n        else:\n            losses = None\n\n        return (all_steps if return_all_steps else predicted_states), losses\n\n\n# ================================================================\n# Patched planning eval for prescribed encoder\n# ================================================================\n\ndef planning_eval_prescribed(plan_cfg, model, env_creator, eval_folder,\n                              num_episodes=10, loader=None, prober=None):\n    """\n    Planning eval that passes locations from env.info to prescribed encoder.\n    \n    Key changes from main_eval:\n    1. set_goal: sets locations from target_position before model.encode\n    2. act loop: sets locations from dot_position before each planning step\n    """\n    plan_cfg = OmegaConf.create(plan_cfg)\n    env = env_creator()\n    env.reset()\n\n    agent = GCAgent(\n        model, action_dim=2, plan_cfg=plan_cfg,\n        normalizer=env.normalizer, loc_prober=prober, env=env,\n    )\n    logger.info(f"Prescribed planning eval with {agent.planner.__class__.__name__}")\n\n    successes = []\n    distances = []\n    episode_times = []\n\n    for ep in range(num_episodes):\n        ep_start = time.time()\n        ep_folder = Path(eval_folder) / f"ep_{ep}"\n        os.makedirs(ep_folder, exist_ok=True)\n\n        obs, info = env.reset()\n        obs, reward, done, truncated, info = env.step(np.zeros(env.action_space.shape[0]))\n        goal_img = info["target_obs"]\n        goal_position = info["target_position"]\n        dot_position = info["dot_position"]\n\n        # --- set_goal with locations ---\n        # For prescribed encoder: set locations before encode\n        goal_loc_tensor = torch.tensor(goal_position, dtype=torch.float32, device=agent.device)\n        # Shape: [2] -> [1, 2, 1] (batch=1, dim=2, time=1)\n        goal_loc_for_enc = goal_loc_tensor.unsqueeze(0).unsqueeze(-1)\n        model.set_locations_for_planning(goal_loc_for_enc)\n        agent.set_goal(goal_img.detach().clone().to(dtype=torch.float32), goal_position)\n        model.clear_planning_locations()\n\n        done = False\n        steps_left = env.n_allowed_steps\n        pbar = tqdm(desc=f"ep {ep}", total=steps_left, leave=True,\n                    disable=plan_cfg.logging.tqdm_silent)\n        t0 = True\n        observations = [obs]\n\n        while steps_left > 0:\n            # Get current dot position from env info\n            dot_position = info["dot_position"]\n            dot_loc_tensor = torch.tensor(dot_position, dtype=torch.float32, device=agent.device)\n            dot_loc_for_enc = dot_loc_tensor.unsqueeze(0).unsqueeze(-1)\n\n            # Set locations for all encode() calls during planning\n            model.set_locations_for_planning(dot_loc_for_enc)\n\n            obs_tensor = (\n                env.normalizer.normalize_state(\n                    obs.detach().clone().to(dtype=torch.float32, device=agent.device)\n                ).unsqueeze(0).unsqueeze(2)\n            )\n            with torch.no_grad():\n                action = agent.act(obs_tensor, steps_left=steps_left, t0=t0).cpu().numpy()\n\n            model.clear_planning_locations()\n\n            for a in action:\n                obs, reward, done, truncated, info = env.step(a)\n                t0 = False\n                observations.append(obs)\n                steps_left -= 1\n                pbar.update(1)\n                eval_results = env.eval_state(info["target_position"], info["dot_position"])\n                success = eval_results["success"]\n                state_dist = eval_results["state_dist"]\n            pbar.set_postfix({"success": success, "dist": f"{state_dist:.3f}"})\n        pbar.close()\n\n        successes.append(success)\n        distances.append(state_dist)\n        ep_time = time.time() - ep_start\n        episode_times.append(ep_time)\n        logger.info(f"  ep {ep}: {\'SUCCESS\' if success else \'FAIL\'} dist={state_dist:.4f} time={ep_time:.0f}s")\n\n    results = {\n        "success_rate": float(np.mean(successes)),\n        "mean_state_dist": float(np.mean(distances)),\n        "avg_episode_time": float(np.mean(episode_times)),\n        "successes": [bool(s) for s in successes],\n        "distances": [float(d) for d in distances],\n    }\n    with open(os.path.join(eval_folder, "planning_results.json"), "w") as f:\n        json.dump(results, f, indent=2)\n    logger.info(f"SR={results[\'success_rate\']:.2f} mean_dist={results[\'mean_state_dist\']:.4f}")\n    return results\n\n\n# ================================================================\n# Build model (same as training)\n# ================================================================\n\nCONDITIONS = {\n    \'free\': {},\n    \'prescribed\': {\'encoder_type\': \'prescribed\'},\n    \'hybrid\': {\'encoder_type\': \'hybrid\'},\n    \'prescribed_no_idm\': {\'encoder_type\': \'prescribed\', \'idm_coeff\': 0},\n    \'prescribed_no_vicreg\': {\'encoder_type\': \'prescribed\', \'std_coeff\': 0, \'cov_coeff\': 0},\n    \'prescribed_no_sim\': {\'encoder_type\': \'prescribed\', \'sim_coeff_t\': 0},\n}\n\n\ndef build_encoder(encoder_type, cfg, data_config):\n    if encoder_type == \'free\':\n        return ImpalaEncoder(\n            width=1,\n            stack_sizes=(16, cfg.model.henc, cfg.model.dstc),\n            num_blocks=2, dropout_rate=None, layer_norm=False,\n            input_channels=cfg.model.dobs, final_ln=True,\n            mlp_output_dim=512,\n            input_shape=(cfg.model.dobs, data_config.img_size, data_config.img_size),\n        )\n    elif encoder_type == \'prescribed\':\n        return PrescribedEncoder(prescribed_dim=2, output_dim=512)\n    elif encoder_type == \'hybrid\':\n        pixel_enc = ImpalaEncoder(\n            width=1,\n            stack_sizes=(16, cfg.model.henc, cfg.model.dstc),\n            num_blocks=2, dropout_rate=None, layer_norm=False,\n            input_channels=cfg.model.dobs, final_ln=True,\n            mlp_output_dim=512,\n            input_shape=(cfg.model.dobs, data_config.img_size, data_config.img_size),\n        )\n        return HybridEncoder(pixel_enc, prescribed_dim=2, prescribed_output_dim=128)\n    else:\n        raise ValueError(f"Unknown encoder_type: {encoder_type}")\n\n\ndef load_model(mode, checkpoint_path, device):\n    """Build model and load checkpoint."""\n    cond = CONDITIONS[mode]\n    encoder_type = cond.get(\'encoder_type\', \'free\')\n    locations_available = encoder_type in (\'prescribed\', \'hybrid\')\n\n    cfg = load_config("examples/ac_video_jepa/cfgs/train.yaml")\n    cfg.data.num_workers = 0\n    cfg.data.pin_mem = False\n    cfg.data.persistent_workers = False\n\n    loader, val_loader, data_config = init_data(\n        env_name=cfg.data.env_name, cfg_data=dict(cfg.data)\n    )\n\n    encoder = build_encoder(encoder_type, cfg, data_config)\n    mlp_dim = encoder.mlp_output_dim\n\n    enc_final_ln = getattr(encoder, \'final_ln\', None)\n    if enc_final_ln is None or isinstance(enc_final_ln, bool):\n        enc_final_ln = nn.LayerNorm(mlp_dim)\n\n    predictor = RNNPredictor(hidden_size=mlp_dim, final_ln=enc_final_ln)\n    aencoder = nn.Identity()\n\n    idm = InverseDynamicsModel(state_dim=mlp_dim, hidden_dim=256, action_dim=2).to(device)\n\n    # Apply ablation overrides\n    reg_cfg = dict(cfg.model.regularizer)\n    for key in (\'idm_coeff\', \'std_coeff\', \'cov_coeff\', \'sim_coeff_t\'):\n        if key in cond:\n            reg_cfg[key] = cond[key]\n\n    regularizer = VC_IDM_Sim_Regularizer(\n        cov_coeff=reg_cfg.get(\'cov_coeff\', cfg.model.regularizer.cov_coeff),\n        std_coeff=reg_cfg.get(\'std_coeff\', cfg.model.regularizer.std_coeff),\n        sim_coeff_t=reg_cfg.get(\'sim_coeff_t\', cfg.model.regularizer.sim_coeff_t),\n        idm_coeff=reg_cfg.get(\'idm_coeff\', cfg.model.regularizer.get("idm_coeff", 0.1)),\n        idm=idm,\n        first_t_only=cfg.model.regularizer.get("first_t_only"),\n        spatial_as_samples=cfg.model.regularizer.spatial_as_samples,\n        idm_after_proj=cfg.model.regularizer.idm_after_proj,\n        sim_t_after_proj=cfg.model.regularizer.sim_t_after_proj,\n    )\n    ploss_fn = SquareLossSeq()\n\n    if locations_available:\n        jepa = PrescribedJEPA(encoder, aencoder, predictor, regularizer, ploss_fn).to(device)\n    else:\n        jepa = JEPA(encoder, aencoder, predictor, regularizer, ploss_fn).to(device)\n\n    xy_head = MLPXYHead(\n        input_shape=mlp_dim,\n        normalizer=loader.dataset.normalizer,\n    ).to(device)\n    xy_prober = JEPAProbe(jepa=jepa, head=xy_head, hcost=nn.MSELoss())\n\n    # Load checkpoint\n    logger.info(f"Loading checkpoint: {checkpoint_path}")\n    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)\n    jepa.load_state_dict(ckpt["model_state_dict"])\n    if "xy_head_state_dict" in ckpt:\n        xy_head.load_state_dict(ckpt["xy_head_state_dict"])\n    epoch = ckpt["epoch"]\n    logger.info(f"Loaded epoch {epoch}")\n    del ckpt\n    torch.cuda.empty_cache()\n\n    return jepa, xy_prober, loader, val_loader, cfg, data_config, epoch, locations_available\n\n\n# ================================================================\n# Main eval function\n# ================================================================\n\ndef run_planning_eval(mode, drive_base, num_episodes=20):\n    """Run planning eval for one condition."""\n    device = torch.device(DEVICE)\n    setup_device("auto")\n    setup_seed(1)\n\n    checkpoint_path = os.path.join(drive_base, mode, "latest.pth.tar")\n    if not os.path.exists(checkpoint_path):\n        logger.error(f"No checkpoint for {mode}: {checkpoint_path}")\n        return None\n\n    jepa, xy_prober, loader, val_loader, cfg, data_config, epoch, locations_available = \\\n        load_model(mode, checkpoint_path, device)\n\n    jepa.eval()\n\n    # Load planning configs\n    with open("examples/ac_video_jepa/cfgs/planning_mppi.yaml") as f:\n        plan_cfg = yaml.safe_load(f)\n    with open("examples/ac_video_jepa/cfgs/eval.yaml") as f:\n        eval_cfg = yaml.safe_load(f)\n\n    plan_cfg["logging"] = {"tqdm_silent": False}\n\n    _, _, env_config = init_data(\n        env_name=cfg.data.env_name, cfg_data=dict(eval_cfg.get("data", {}))\n    )\n\n    def env_creator():\n        from eb_jepa.datasets.two_rooms.env import DotWall\n        return DotWall(config=env_config, **eval_cfg.get("env", {}))\n\n    eval_folder = Path(os.path.join(drive_base, mode, f"planning_eval_ep{epoch}"))\n    os.makedirs(eval_folder, exist_ok=True)\n\n    logger.info(f"=== Planning eval: {mode} (epoch {epoch}) ===")\n    logger.info(f"  locations_available={locations_available}, device={device}")\n    logger.info(f"  num_episodes={num_episodes}")\n\n    if locations_available:\n        # Use patched planning eval for prescribed/hybrid\n        results = planning_eval_prescribed(\n            plan_cfg=plan_cfg, model=jepa, env_creator=env_creator,\n            eval_folder=eval_folder, num_episodes=num_episodes,\n            loader=val_loader, prober=xy_prober,\n        )\n    else:\n        # Use original planning eval for free\n        results = main_eval(\n            plan_cfg=plan_cfg, model=jepa, env_creator=env_creator,\n            eval_folder=eval_folder, num_episodes=num_episodes,\n            loader=val_loader, prober=xy_prober,\n        )\n\n    # Save summary\n    results[\'mode\'] = mode\n    results[\'epoch\'] = epoch\n    results[\'device\'] = str(device)\n    summary_path = os.path.join(drive_base, mode, "planning_eval_results.json")\n    with open(summary_path, "w") as f:\n        json.dump(results, f, indent=2)\n    logger.info(f"Results saved to {summary_path}")\n\n    return results\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", type=str, required=True, choices=list(CONDITIONS.keys()))\n    parser.add_argument("--drive_base", type=str, default="/content/drive/MyDrive/eb_jepa_v3_clean")\n    parser.add_argument("--num_episodes", type=int, default=20)\n    args = parser.parse_args()\n    run_planning_eval(args.mode, args.drive_base, args.num_episodes)\n'

with open(script_path, 'w') as f:
    f.write(SCRIPT)
print(f'Written: {script_path} ({os.path.getsize(script_path)} bytes)')


In [ ]:
# CELL 4: Check checkpoints
import os, torch
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
for mode in ['free', 'prescribed']:
    ckpt = os.path.join(DRIVE_BASE, mode, 'latest.pth.tar')
    if os.path.exists(ckpt):
        c = torch.load(ckpt, map_location='cpu', weights_only=False)
        print(f'{mode}: epoch {c["epoch"]}, batch_idx={c.get("batch_idx", "N/A")}')
        del c
    else:
        print(f'{mode}: NO CHECKPOINT')

In [ ]:
# CELL 5: Run planning eval — FREE
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
!cd /content/eb_jepa && python experiments/prescribed_axes/planning_eval.py --mode free --drive_base {DRIVE_BASE} --num_episodes 20

In [ ]:
# CELL 6: Run planning eval — PRESCRIBED
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
!cd /content/eb_jepa && python experiments/prescribed_axes/planning_eval.py --mode prescribed --drive_base {DRIVE_BASE} --num_episodes 20

In [ ]:
# CELL 7: Results
import json, os
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'

print('=' * 60)
print(f'{"Mode":<20} {"SR":>6} {"Mean Dist":>10} {"Avg Time":>10}')
print('-' * 60)
for mode in ['free', 'prescribed']:
    rp = os.path.join(DRIVE_BASE, mode, 'planning_eval_results.json')
    if os.path.exists(rp):
        with open(rp) as f:
            r = json.load(f)
        print(f'{mode:<20} {r["success_rate"]:>6.2f} {r["mean_state_dist"]:>10.4f} {r.get("avg_episode_time", 0):>9.0f}s')
    else:
        print(f'{mode:<20} {"--":>6}')
print('=' * 60)

In [ ]:
# CELL 8: Download
from google.colab import files
import shutil
DRIVE_BASE = '/content/drive/MyDrive/eb_jepa_v3_clean'
collect = '/content/planning_eval_download'
if os.path.exists(collect):
    shutil.rmtree(collect)
os.makedirs(collect)
for mode in os.listdir(DRIVE_BASE):
    md_path = os.path.join(DRIVE_BASE, mode)
    if not os.path.isdir(md_path):
        continue
    dst = os.path.join(collect, mode)
    os.makedirs(dst, exist_ok=True)
    for fn in ['results.json', 'planning_eval_results.json', 'encoder_stats.json']:
        src = os.path.join(md_path, fn)
        if os.path.exists(src):
            shutil.copy2(src, dst)
shutil.make_archive('/content/planning_eval_results', 'zip', collect)
files.download('/content/planning_eval_results.zip')